In [ ]:
import os

os.chdir("..")
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import torch
from dotenv import load_dotenv

load_dotenv()

# Scale model ready

In [ ]:
data_dir = os.path.join(os.environ["DATA_DIR"]) + "/satbird-USA-summer/"
scales = pd.read_csv(data_dir + "bioclim_classes.csv")
scales.head()

In [ ]:
df = pd.read_csv(data_dir + "model_ready_satbird-USA-summer_with_lc.csv")
df.head()

In [ ]:
for col in scales.name:
    scale = scales[scales.name == col]["scale"].item()
    df[col] = df[col].apply(lambda i: i * scale)
df.head()

if not os.path.exists(data_dir + "model_ready_satbird-USA-summer_with_lc_scaled.csv"):
    df.to_csv(data_dir + "model_ready_satbird-USA-summer_with_lc_scaled.csv")

# Make a list of only train and val files to copy

In [ ]:
pth = "data/satbird-USA-summer/splits/satbird-USA-summer_aef_union_val_test_lc.pth"
split_indices = torch.load(pth, weights_only=False)
lst = list(split_indices["test_indices"]) + list(split_indices["val_indices"])

In [ ]:
with open("files_to_copy.txt", "w") as f:
    for id_ in lst:
        f.write(f"s2_{id_}.tif\n")

```bash
 rsync --files-from=Desktop/aether/files_to_copy.txt --ignore-existing
--stats -avzh --progress
/Volumes/KINGSTON/data/satbird-USA-summer/eo/s2
gtijunaityte@snellius.surf.nl:/gpfs/home2/gtijunaityte/aether/data/satbird-USA-summer/eo/
```

# Visualisation of crops


In [ ]:
def load_tiff(tiff_file_path: str, dtype: np.dtype) -> np.ndarray:
    """Load tiff file as np array of a specified dtype."""

    with rasterio.open(tiff_file_path) as f:
        im = f.read()
        assert isinstance(im, np.ndarray)
        if im.dtype != np.dtype(dtype):
            im = im.astype(dtype=dtype, copy=False)
    return im


import matplotlib.pyplot as plt
import numpy as np


def plot_arr(arr):
    rgb = arr[[2, 1, 0]]  # e.g. B4,B3,B2 -> R,G,B order
    rgb = np.transpose(rgb, (1, 2, 0))  # (H, W, 3)

    # normalize for display if reflectance/DN values
    rgb = np.clip(rgb / rgb.max(), 0, 1)
    plt.imshow(rgb)
    plt.axis("off")
    plt.show()

In [ ]:
lst = list(split_indices["test_indices"])

In [ ]:
from utils.data_utils import center_crop_npy

path = "/Volumes/KINGSTON/data/satbird-USA-summer/eo/s2/s2_L1006834.tif"
im = load_tiff(path, dtype=np.dtype("uint16"))
im = im[[2, 1, 0], :, :]
c = 3
plot_arr(im)
# Crop
size = 224
if im.shape[-2:] != (size, size):
    im = center_crop_npy(im, (c, size, size))
plot_arr(im)

In [ ]:
plt.show()
import numpy as np


def center_crop_or_pad_npy(arr, target_shape):
    """Center-crops dims larger than target, center-pads (zeros) dims smaller than target."""
    if len(arr.shape) != len(target_shape):
        raise ValueError(f"arr has {len(arr.shape)} dims but target_shape has {len(target_shape)}")

    pad_widths = []
    for dim, target in zip(arr.shape, target_shape):
        if dim < target:
            total_pad = target - dim
            before = total_pad // 2
            after = total_pad - before
            pad_widths.append((before, after))
        else:
            pad_widths.append((0, 0))
    arr = np.pad(arr, pad_widths, mode="constant", constant_values=0)

    slices = []
    for dim, target in zip(arr.shape, target_shape):
        start = (dim - target) // 2  # 0 if dim == target
        end = start + target
        slices.append(slice(start, end))
    return arr[tuple(slices)]

In [ ]:
im = center_crop_or_pad_npy(im, (3, 224, 224))
plot_arr(im)